In [0]:
from pyspark.sql.functions import lit, col
from pyspark.sql.types import *

def align_schema(src_df, target_table):
    #new table creation, return as is
    if not spark.catalog.tableExists(target_table):
        print("Target table does not exist → initial load")
        return src_df

    tgt_df = spark.table(target_table)
    tgt_schema = tgt_df.schema

    src_cols = set(src_df.columns)
    tgt_cols = set([f.name for f in tgt_schema])

    #Add missing columns in source
    missing_in_src = tgt_cols - src_cols
    for c in missing_in_src:
        dtype = [f.dataType for f in tgt_schema if f.name == c][0]
        src_df = src_df.withColumn(c, lit(None).cast(dtype))

    #datatype casting
    for field in tgt_schema:
        if field.name in src_df.columns:
            src_df = src_df.withColumn(
                field.name,
                col(field.name).cast(field.dataType)
            )
    #Add new columns and sort the columns order
    new_cols = src_cols - tgt_cols
    final_cols = [f.name for f in tgt_schema] + list(new_cols)
    src_df = src_df.select(*final_cols)
    return src_df